In [ ]:
#Binary classifier using a deep learning neural network

In [ ]:
#normal imports and load our csv file into a dataframe
import numpy as np
import pandas as pd

mushrooms_df = pd.read_csv('agaricus-lepiota.csv')
mushrooms_df

In [ ]:
#check the dataframe for missing values
mushrooms_df.info()

In [ ]:
#split our original dataframe into a features set and a target set. Our target is column p and all other columns are my features.
features_df = mushrooms_df.drop('p', axis=1)
target_df = mushrooms_df['p']

In [ ]:
#check our target df
target_df.head()

In [ ]:
#check our features df
features_df.head()

In [ ]:
#import for creating our training and test sets. Then create our training and test sets with 80% training size.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(features_df, target_df, train_size=0.8)

In [ ]:
#load our imports for onehotencodeing and labelencoding
#In this cell we will labelencode our feature column
#we do this by creating our label encoder object, then calling on the object to fit and transform the data

from sklearn.preprocessing import OneHotEncoder, LabelEncoder

label_encoder = LabelEncoder()

#y_train_label_encoded = y_train
y_train_label_encoded = label_encoder.fit_transform(y_train)


#y_test_label_encoded = y_test
y_test_label_encoded = label_encoder.fit_transform(y_test)

In [ ]:
#Print some values to ensure our label encoder did in fact label encode

print(y_train_label_encoded[0:9])
print(y_test_label_encoded[0:9])

In [ ]:
#We will do the one hot encoding in this cell
#I will set drop to 'first' for best practice and sparse = False to get my dense array
#After creating the object for the onehotencoder, use it and call fit_transform on the training data and just transform on the test data
#I do this because I want the test data to be encoded in the same way as the test data. Calling fit.transform on the test data gave me
# 92 columns instead of 94 like the test data. Using transform created the necessary 94 columns. 
one_hot_encoder = OneHotEncoder(sparse = False, drop = 'first')

#X_train_hot_encoded = X_train
X_train_hot_encoded = one_hot_encoder.fit_transform(X_train)

#X_test_hot_encoded = X_test
X_test_hot_encoded = one_hot_encoder.transform(X_test)

In [ ]:
#create our neural network for binary classifiers. 
# I instantiate my model with the Sequential class
# I create my first layer, making sure my input_dim is equal to my new number of features after one hot encoding
# I use the standard output layer with 1 neuron and activation sigmoid
# I use the standard compile parameters for creating a binary classifier. 
# I use the .summary() to check my network.
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()

model.add(Dense(128, activation='relu', input_dim=94))

model.add(Dense(1,activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
%%time
hist = model.fit(X_train_hot_encoded, y_train_label_encoded, validation_data=(X_test_hot_encoded, y_test_label_encoded), epochs=10, batch_size=100)

# In order to fit my model, I use all of the encoded data, run with 10 epochs, and decide on a batch size of 100.

In [ ]:
#plotting our training accuracy versus our validation accuracy

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

acc = hist.history['accuracy']
val = hist.history['val_accuracy']
epochs = range(1, len(acc) + 1)

plt.plot(epochs, acc, '-', label='Training accuracy')
plt.plot(epochs, val, ':', label='Validation accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.plot()

In [ ]:
#creating our predictions for our confusion matrix and scores

from sklearn.metrics import ConfusionMatrixDisplay as cmd
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, ConfusionMatrixDisplay


sns.reset_orig()
y_predicted = model.predict(X_test_hot_encoded) > 0.5

In [ ]:
#use our predicted scores versus our y_test encoded data to calculate our scores
# use our predictions versus our y_test encoded data to create our confusion matrix

accuracy = accuracy_score(y_test_label_encoded, y_predicted)
recall = recall_score(y_test_label_encoded, y_predicted)
f1 = f1_score(y_test_label_encoded, y_predicted)
precision = precision_score(y_test_label_encoded, y_predicted)

print(f"Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")


labels = ['Negative', 'Positive']
cmd.from_predictions(y_test_label_encoded, y_predicted, display_labels=labels, cmap='Blues', xticks_rotation='vertical')

In [ ]:
#The model performed perfectly according to our confusion matrix

In [ ]:
#create a pca model containing 95% if the variation
# In order to do this we use the n_components = paramenter and set it equal to 0.95
# Then we call fit_transform on the training data and just transform on the test data
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)
pca_X_train = pca.fit_transform(X_train_hot_encoded)
pca_X_test =  pca.transform(X_test_hot_encoded)

In [ ]:
#create our neural network for our pca data, we follow the same standards as the above neural network. What changes is our input_dims because
#the pca features are less than before. 
pca_model = Sequential()

pca_model.add(Dense(128, activation='relu', input_dim=38))

pca_model.add(Dense(1,activation='sigmoid'))

pca_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
pca_model.summary()

In [ ]:
%%time
pca_hist = pca_model.fit(pca_X_train, y_train_label_encoded, validation_data=(pca_X_test, y_test_label_encoded), epochs=10, batch_size=100)
#fit our pca_model with our data, our x data is now the pca x data

In [ ]:
#plotting our training accuracy versus our validation accuracy

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

acc = pca_hist.history['accuracy']
val = pca_hist.history['val_accuracy']
epochs = range(1, len(acc) + 1)

plt.plot(epochs, acc, '-', label='Training accuracy')
plt.plot(epochs, val, ':', label='Validation accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.plot()

In [ ]:
sns.reset_orig()
pca_y_predicted = pca_model.predict(pca_X_test) > 0.5

#use our predicted scores versus our y_test encoded data to calculate our scores
# use our predictions versus our y_test encoded data to create our confusion matrix

accuracy = accuracy_score(y_test_label_encoded, pca_y_predicted)
recall = recall_score(y_test_label_encoded, pca_y_predicted)
f1 = f1_score(y_test_label_encoded, pca_y_predicted)
precision = precision_score(y_test_label_encoded, pca_y_predicted)

print(f"Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")


labels = ['Negative', 'Positive']
cmd.from_predictions(y_test_label_encoded, y_predicted, display_labels=labels, cmap='Blues', xticks_rotation='vertical')

In [ ]:
import pickle

with open('model.pkl', 'wb') as file:
    pickle.dump(pca_model , file)   